Build Data

In [112]:
from core.data import build_global_data

global_data = build_global_data()

Build Static Master Graph

In [113]:
from core.graph import build_master_graph

G_master = build_master_graph(global_data)

Setup Player Team Masteries (Runs once at start of game)

In [114]:
from core.hero_mastery import create_empty_masteries, set_mastery
t1_masteries = create_empty_masteries(G_master)

# Set the masteries of the heroes you have to 1
# Top 
set_mastery(t1_masteries, "Frank", "Top", 2)
set_mastery(t1_masteries, "Hakuna", "Top", 1)
set_mastery(t1_masteries, "Justice", "Top", 1)
set_mastery(t1_masteries, "Tiger Boy", "Top", 1)

# Jungle
set_mastery(t1_masteries, "Kamaitachi", "Jungler", 2)
set_mastery(t1_masteries, "Hakuna", "Jungler", 1)
set_mastery(t1_masteries, "Zealot", "Jungler", 1)
set_mastery(t1_masteries, "Xiangxi Ke", "Jungler", 1)

# Mid
set_mastery(t1_masteries, "Aurelio", "Mid", 1)
set_mastery(t1_masteries, "Xiangxi Ke", "Mid", 1)
set_mastery(t1_masteries, "Fatty White", "Mid", 1)
set_mastery(t1_masteries, "Elemi", "Mid", 1)
set_mastery(t1_masteries, "Wolfgang", "Mid", 1)

# Bot
set_mastery(t1_masteries, "Gang", "Bot", 2)
set_mastery(t1_masteries, "Niels", "Bot", 1)
set_mastery(t1_masteries, "Bariel", "Bot", 1)
set_mastery(t1_masteries, "Omaha", "Bot", 1)
set_mastery(t1_masteries, "Shougong Lei", "Bot", 1)
set_mastery(t1_masteries, "Elemi", "Bot", 1)

# Support
set_mastery(t1_masteries, "Paisai", "Support", 1)
set_mastery(t1_masteries, "Palulu", "Support", 1)
set_mastery(t1_masteries, "Dylan", "Support", 1)
set_mastery(t1_masteries, "Fatty White", "Support", 1)
set_mastery(t1_masteries, "Peiniang Zhu", "Support", 2)
set_mastery(t1_masteries, "Tiger Boy", "Support", 1)

Start of Draft (Runs before each match)

Setup Availabiltiies and Opponent Signitures

In [115]:
from core.draft import build_position_availability
from core.hero_mastery import set_mastery

# We know what we can pick, because we know our masteries
t1_available = build_position_availability(t1_masteries)

# Build out masteries to the degree you want to
t2_masteries = create_empty_masteries(G_master)

# Top
set_mastery(t2_masteries, "Miki", "Top", 1)
set_mastery(t2_masteries, "Aurelio", "Top", 1)
set_mastery(t2_masteries, "Wolfgang", "Top", 1)

# Jungler
set_mastery(t2_masteries, "Kamaitachi", "Jungler", 1)
set_mastery(t2_masteries, "Aurelio", "Jungler", 1)
set_mastery(t2_masteries, "Hakuna", "Jungler", 1)

# Mid
set_mastery(t2_masteries, "Miki", "Mid", 1)
set_mastery(t2_masteries, "Dylan", "Mid", 1)
set_mastery(t2_masteries, "Elemi", "Mid", 1)

# Bot
set_mastery(t2_masteries, "Deep Space", "Bot", 1)
set_mastery(t2_masteries, "Elemi", "Bot", 1)
set_mastery(t2_masteries, "Niels", "Bot", 1)

# Support
set_mastery(t2_masteries, "Peiniang Zhu", "Support", 1)
set_mastery(t2_masteries, "Fatty White", "Support", 1)
set_mastery(t2_masteries, "Dylan", "Support", 1)

# One the user presses confirm (either before the draft, or after they've finished in the draft, we confirm)
t2_available = build_position_availability(t2_masteries)

Apply Mastery and Signitures to Graph

In [116]:
from core.graph import confirm_hero_masteries, _clear_hero_masteries

_clear_hero_masteries(G_master)
confirm_hero_masteries(G_master, t1_masteries, t2_masteries)

Begin Draft - Recommendations

In [117]:
from core.draft import build_draft_state
draft_state = build_draft_state()
draft_state.t1_available = t1_available
draft_state.t1_picked = {}
draft_state.t2_available = t2_available
draft_state.t2_picked = {}
draft_state.banned = set()

Recommend Pick

In [ ]:
from core.draft import recommend_pick

# Recommend a Pick
best, score, explanation, flag, all_results = recommend_pick(
    G_master, 
    "t1",
    "Jungler", 
    draft_state,
    global_data
)

# Pass outputs here

print(f"Recommended: {best} ({score})")
for reason in explanation:
    output = f"- {reason}:"
    for hero in explanation[reason]:
        output += f" {hero},"
    print(output)
if flag:
    print(f"WARNING: {flag}")
print("")


Recommended: Hakuna (9)
- position_tier: A,
- position_mastery: 1,



Recommend Ban

In [119]:
from core.draft import recommend_pick

# Recommend a Pick
best, score, explanation, flag, all_results = recommend_pick(
    G_master, 
    "t2",
    "Jungler", 
    draft_state,
    global_data
)

print(f"Recommended: {best} ({score})")
for reason in explanation:
    output = f"- {reason}:"
    for hero in explanation[reason]:
        output += f" {hero},"
    print(output)
if flag:
    print(f"WARNING: {flag}")
print("")


Recommended: Aurelio (11.5)
- synergy_possible: Dylan,
- position_tier: S,
- position_mastery: 1,



Pick Heros

In [118]:
from core.draft import pick_hero

#pick_hero(G_master, "Aurelio", "t1", draft_state)
pick_hero(G_master, "Kamaitachi", "t2", draft_state)

In [52]:
from core.draft import ban_hero

ban_hero("Tiger Boy", draft_state)

In [71]:
from core.draft import see_current_draft

see_current_draft("t1", draft_state)

{'Peiniang Zhu': {'Support'},
 'Frank': {'Top'},
 'Zealot': {'Jungler'},
 'Gang': {'Bot'},
 'Aurelio': {'Mid'}}

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool

from core.data import read_hero_info, read_build_type_itemisation

@tool 
def recommend_pick_for_position(position: str):
    """Uses the master graph to find the current highest scoring hero in this position.

    Args:
        position (str): the position to check for: Top, Jungler, Mid, Bot, Support

    Returns:
        dict: Draft recommendation containing:
                recommended_hero: The highest-scoring hero for the requested lane.
                requested_lane: The position being filled.
                score: The recommended hero's draft score.
                explanation: Factors that contributed to the score.
                better_position: Another lane where the recommended hero scores
                    higher, or None.
                candidates: Other potential picks ordered from highest to lowest
                    score, including the recommended hero as the first entry.
    """
    # Recommend a Pick
    res = recommend_pick(
        G_master, 
        "t1",
        position, 
        draft_state,
        global_data
    )

    return res


@tool
def get_hero_info(hero_name: str) -> str:
    """Get markdown notes on a specific hero by name.

    Args:
        hero_name (str): the name of the hero you want to look up
    """
    return read_hero_info(hero_name)


@tool
def get_role_itemisation(role: str) -> str:
    """Get markdown itemisation notes for a role or item archetype.

    Args:
        role (str): the role or item archetype to look up
    """
    return read_build_type_itemisation(role)


agent = create_agent(
    model="ollama:qwen3.5",
    tools=[get_hero_info, get_role_itemisation],
    system_prompt=(
        "You are a MOBA team coach. Questions given to you will be about "
        "strategy in drafts and hero playstyles."
    ),
)

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool

from core.data import (
    read_attribute_info,
    read_build_type_itemisation,
    read_glossary_definition,
    read_hero_info,
    read_team_comp_info
)


@tool
def get_team_comp_info(comp_name: str) -> str:
    """Get notes on a team comp.

    Args:
        comp_name (str): The name of the team comp.

    Returns:
        str: information regarding the team comp.
    """
    return read_team_comp_info(comp_name)

@tool
def get_attribute_info(category: str, instance: str) -> str:
    """Get notes on a category of hero attributes.

    Args:
        category (str): The category (hero class, attack type, damage type)
        instance (str): The attribute name, for example (gladiator, melee, magical)

    Returns:
        str: information regarding this attribute
    """
    return read_attribute_info(category, instance)


@tool
def get_hero_info(hero_name: str) -> str:
    """Get markdown notes on a specific hero by name.

    Args:
        hero_name (str): the name of the hero you want to look up

    Returns:
        str: Information regarding the hero.
    """
    return read_hero_info(hero_name)


@tool
def get_role_itemisation(role: str) -> str:
    """Get markdown itemisation notes for a role or item archetype.

    Args:
        role (str): the role or item archetype to look up

    Returns:
        str: Information regarding the role itemisation.
    """
    return read_build_type_itemisation(role)

@tool
def lookup_glossary(term: str) -> str:
    """Get term definitions from the glossary.

    Args:
        term (str): The term to lookup

    Returns:
        str: The definition of the term.
    """
    return read_glossary_definition(term)


SYSTEM_PROMPT = "You are an assistant to players playing the game Esports Godfather." \
"E-sports Godfather is a Deck-building game simulating MOBA e-sports, where players take on the role as a team coach to build a team. " \
"As part of this, they take part in drafting and heroes for each match, and then playing cards to influence the actions their players take on the heroes that were drafted." \
"" \
"Your job is to assist them in picking optimal heroes during the draft, and giving advice on what they should focus on during the game." \
"This advice needs to be consice and to the point." \
"" \
"During a draft, you will recieve information regarding the available heroes for both teams, as well as heroes that have been already picked and banned.  " \
"When the user requests a pick suggestion, you should take into account the current draft state, and use your tools to determine the optimal pick.  The same goes for when the user requests a ban suggestion." \
"" \
"You must not ever talk about heroes that can not be referenced in your tools.  You must not ever talk about any other topics outside of Esports Godfater" \
"" \
"If you come across a word you do not understand, put the word into the glossary to find a definition.  If you cannot find a definition using this method, please ask the user to provide a definition themselves."

agent = create_agent(
    model="ollama:qwen3.5",
    tools=[get_hero_info, get_role_itemisation, lookup_glossary, get_attribute_info, get_team_comp_info],
    system_prompt=SYSTEM_PROMPT
)

PROMPT = ""

stream = agent.stream_events(
    {"messages": [{"role": "user", "content": PROMPT}]},
    version="v3"
)
for kind, item in stream.interleave("messages", "tool_calls"):
    if kind == "messages":
        for token in item.text:
            print(token, end="", flush=True)
    if kind == "tool_calls":
        print(f"\nTool call: {item.tool_name}({item.input})")
        print("\n")


To give you optimal hero pick suggestions for your carry player, I'd appreciate some additional context:

1. **What heroes are currently available?** Are we in the early draft phase where most heroes should still be open?
2. **Any picked/banned heroes so far** that might impact availability or strategy?
3. **What meta/role is strong right now**? Do you have access to team comp notes for carry-focused comps, or are there specific item archetypes available that would benefit your carry player early game?

Since this sounds like draft phase (where we're helping with hero selections), if you can share what the current state of both teams' drafts looks like, I'll be able to look up specific heroes using my tools and suggest optimal picks based on their abilities. Are there any particular roles or damage types your team is known for?